In [ ]:
import pandas as pd

URL = "https://raw.githubusercontent.com/yamanalab/ec-darkpattern/master/dataset/dataset.tsv"
df = pd.read_csv(URL, sep="\t")
print(df.shape)
print(df.columns.tolist())
df.head(10)

(2356, 4)
['page_id', 'text', 'label', 'Pattern Category']


,page_id,text,label,Pattern Category
0,1012,FLASH SALE | LIMITED TIME ONLY Shop Now,1,Urgency
1,158,Pillowcases & Shams,0,Not Dark Pattern
2,108,Write a review,0,Not Dark Pattern
3,1425,"To start your return, simply click on the foll...",0,Not Dark Pattern
4,1658,newsletter signup (privacy policy),0,Not Dark Pattern
5,975,In demand,1,Scarcity
6,1469,We cannot guarantee door to door delivery for ...,0,Not Dark Pattern
7,1570,Hurry! Only 2 left in stock,1,Scarcity
8,1617,In Stock only 3 left,1,Scarcity
9,108,International Shipping Policy,0,Not Dark Pattern


In [ ]:
TEXT_COL = "text"
LABEL_COL = "label"
CAT_COL = "Pattern Category"
GROUP_COL = "page_id"

df = pd.read_csv(URL, sep="\t")
df = df[[TEXT_COL, LABEL_COL, CAT_COL, GROUP_COL]].dropna()
df[TEXT_COL] = df[TEXT_COL].astype(str).str.strip()
df = df[df[TEXT_COL] != ""]
df = df.drop_duplicates(subset=TEXT_COL)
df[LABEL_COL] = df[LABEL_COL].astype(int)

print(df.shape)
print("\nTexts per category:")
print(df[CAT_COL].value_counts())

df["words"] = df[TEXT_COL].str.split().str.len()
print("\nWords per text, by label:")
print(df.groupby(LABEL_COL)["words"].describe())

(2349, 4)

Texts per category:
Pattern Category
Not Dark Pattern    1171
Scarcity             418
Social Proof         312
Urgency              210
Misdirection         195
Obstruction           27
Sneaking              12
Forced Action          4
Name: count, dtype: int64

Words per text, by label:
        count      mean        std  min  25%  50%  75%    max
label                                                        
0      1171.0  5.764304   8.852879  1.0  2.0  3.0  6.0  116.0
1      1178.0  9.105263  12.892568  1.0  4.0  6.0  9.0  143.0


In [ ]:
from sklearn.model_selection import GroupShuffleSplit

gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
train_idx, temp_idx = next(gss.split(df, df[LABEL_COL], df[GROUP_COL]))
train_df = df.iloc[train_idx]
temp_df = df.iloc[temp_idx]

gss2 = GroupShuffleSplit(n_splits=1, test_size=0.5, random_state=42)
val_idx, test_idx = next(gss2.split(temp_df, temp_df[LABEL_COL], temp_df[GROUP_COL]))
val_df = temp_df.iloc[val_idx]
test_df = temp_df.iloc[test_idx]

# Proof that no website appears in two sets
assert set(train_df[GROUP_COL]).isdisjoint(set(test_df[GROUP_COL]))
assert set(train_df[GROUP_COL]).isdisjoint(set(val_df[GROUP_COL]))

for name, d in [("train", train_df), ("val", val_df), ("test", test_df)]:
    print(name, len(d), "rows,", d[GROUP_COL].nunique(), "websites, dark share:", round(d[LABEL_COL].mean(), 2))

train 1609 rows, 873 websites, dark share: 0.51
val 354 rows, 187 websites, dark share: 0.5
test 386 rows, 188 websites, dark share: 0.45


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

# Ordinary short UI text the model should learn is NOT manipulative
NEG_TRAIN = [
    "No thanks", "No, thanks", "Not now", "Maybe later", "Skip", "Cancel", "Close",
    "Decline", "No", "Continue shopping", "Add to cart", "Add to bag", "Buy now",
    "Checkout", "Proceed to checkout", "Sign in", "Sign up", "Log in", "Create account",
    "My account", "Contact us", "About us", "Help", "FAQ", "Search", "Shop all",
    "New arrivals", "View details", "Learn more", "Read more", "Submit", "Send", "Save",
    "Next", "Back", "Continue", "Confirm", "Apply", "Update cart", "Remove",
    "Your cart is empty", "Shipping information", "Return policy", "Privacy policy",
    "Terms and conditions", "Track your order", "Gift cards", "Size guide", "Reviews",
    "Wishlist", "Subscribe", "Email address", "Accept cookies", "Reject cookies",
    "Manage preferences", "Free shipping on orders over 50",
]

# Kept OUT of training, used only to check the model generalises
NEG_HELDOUT = [
    "No thank you", "Not right now", "Remind me later", "Close window", "Keep browsing",
    "View cart", "Add to wishlist", "Save for later", "Proceed to payment",
    "Enter your email address", "Customer support", "Order history",
]

# Repeat each x3 so the model notices them among ~1,600 training rows
neg_df = pd.DataFrame({TEXT_COL: NEG_TRAIN * 3, LABEL_COL: 0})
train_plus = pd.concat([train_df[[TEXT_COL, LABEL_COL]], neg_df], ignore_index=True)

model = Pipeline([
    ("tfidf", TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)),
    ("clf", LogisticRegression(max_iter=1000, class_weight="balanced")),
])
model.fit(train_plus[TEXT_COL], train_plus[LABEL_COL])
print("trained on", len(train_plus), "rows")

trained on 1777 rows


In [ ]:
import numpy as np
from sklearn.metrics import precision_score, recall_score, f1_score, classification_report, confusion_matrix

probs = model.predict_proba(val_df[TEXT_COL])[:, 1]
y_val = val_df[LABEL_COL]

best = None
print("threshold  precision  recall    f1")
for t in np.arange(0.30, 0.91, 0.05):
    pred = (probs >= t).astype(int)
    p = precision_score(y_val, pred, zero_division=0)
    r = recall_score(y_val, pred)
    f = f1_score(y_val, pred)
    print(f"{t:.2f}       {p:.3f}      {r:.3f}     {f:.3f}")
    if p >= 0.95 and (best is None or f > best[1]):
        best = (t, f)

chosen_threshold = round(best[0], 2) if best else 0.5
print("\nChosen threshold (best F1 while precision >= 0.95):", chosen_threshold)

threshold  precision  recall    f1
0.30       0.856      0.972     0.910
0.35       0.905      0.966     0.934
0.40       0.918      0.955     0.936
0.45       0.938      0.944     0.941
0.50       0.958      0.893     0.924
0.55       0.981      0.870     0.922
0.60       0.980      0.842     0.906
0.65       0.993      0.808     0.891
0.70       0.993      0.757     0.859
0.75       0.992      0.695     0.817
0.80       1.000      0.582     0.736
0.85       1.000      0.503     0.669
0.90       1.000      0.379     0.549

Chosen threshold (best F1 while precision >= 0.95): 0.5


In [ ]:
THRESHOLD = chosen_threshold

test_probs = model.predict_proba(test_df[TEXT_COL])[:, 1]
test_pred = (test_probs >= THRESHOLD).astype(int)
y_test = test_df[LABEL_COL]

print("Threshold:", THRESHOLD)
print(classification_report(y_test, test_pred, digits=3))
cm = confusion_matrix(y_test, test_pred)
print(cm)
tn, fp, fn, tp = cm.ravel()
print("false positive rate:", round(fp / (fp + tn), 3))

Threshold: 0.5
              precision    recall  f1-score   support

           0      0.931     0.957     0.944       211
           1      0.947     0.914     0.930       175

    accuracy                          0.938       386
   macro avg      0.939     0.936     0.937       386
weighted avg      0.938     0.938     0.938       386

[[202   9]
 [ 15 160]]
false positive rate: 0.043


In [ ]:
print("Held-out ordinary UI text (want LOW):")
for s, p in zip(NEG_HELDOUT, model.predict_proba(NEG_HELDOUT)[:, 1]):
    print(f"{p:.2f}  {s}")

positives = [
    "Only 2 left in stock!", "Hurry! Offer ends in 09:59",
    "12 people are viewing this right now", "Last chance - sale ends tonight",
    "No thanks, I hate saving money", "No, I don't want to save money",
    "No thanks, I'd rather pay full price",
]
print("\nSuspicious text (want HIGH):")
for s, p in zip(positives, model.predict_proba(positives)[:, 1]):
    print(f"{p:.2f}  {s}")

Held-out ordinary UI text (want LOW):
0.36  No thank you
0.41  Not right now
0.17  Remind me later
0.15  Close window
0.21  Keep browsing
0.15  View cart
0.19  Add to wishlist
0.34  Save for later
0.19  Proceed to payment
0.20  Enter your email address
0.18  Customer support
0.27  Order history

Suspicious text (want HIGH):
0.99  Only 2 left in stock!
0.89  Hurry! Offer ends in 09:59
0.85  12 people are viewing this right now
0.65  Last chance - sale ends tonight
0.53  No thanks, I hate saving money
0.79  No, I don't want to save money
0.92  No thanks, I'd rather pay full price


In [ ]:
dark_train = train_df[train_df[LABEL_COL] == 1]
dark_test = test_df[test_df[LABEL_COL] == 1]

cat_model = Pipeline([
    ("tfidf", TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)),
    ("clf", LogisticRegression(max_iter=1000, class_weight="balanced")),
])
cat_model.fit(dark_train[TEXT_COL], dark_train[CAT_COL])

print(classification_report(
    dark_test[CAT_COL], cat_model.predict(dark_test[TEXT_COL]),
    digits=3, zero_division=0))

              precision    recall  f1-score   support

Misdirection      0.963     0.867     0.912        30
 Obstruction      0.833     1.000     0.909         5
    Scarcity      1.000     0.948     0.973        58
    Sneaking      0.000     0.000     0.000         1
Social Proof      0.911     1.000     0.953        51
     Urgency      0.903     0.933     0.918        30

    accuracy                          0.943       175
   macro avg      0.768     0.791     0.778       175
weighted avg      0.941     0.943     0.940       175



In [ ]:
samples = [
    "Add to cart", "Contact us", "Sign in", "No thanks", "Your cart is empty",
    "Free delivery on orders above 499", "Subscribe to our newsletter",
    "Only 2 left in stock!", "Hurry! Offer ends in 09:59",
    "12 people are viewing this right now",
    "No thanks, I hate saving money", "Yes, send me offers from partners",
]
for s, p in zip(samples, model.predict_proba(samples)[:, 1]):
    print(f"{p:.2f}  {s}")

0.36  Add to cart
0.27  Contact us
0.28  Sign in
0.94  No thanks
0.36  Your cart is empty
0.37  Free delivery on orders above 499
0.19  Subscribe to our newsletter
0.99  Only 2 left in stock!
0.89  Hurry! Offer ends in 09:59
0.86  12 people are viewing this right now
0.78  No thanks, I hate saving money
0.32  Yes, send me offers from partners


In [ ]:
import joblib
import sklearn
from google.colab import files

print("scikit-learn version:", sklearn.__version__)   # write this down
joblib.dump({
    "pipeline": model,
    "threshold": THRESHOLD,
    "category_pipeline": cat_model,
}, "model.joblib")
files.download("model.joblib")

scikit-learn version: 1.6.1


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>